# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data


A model that can estimate how much something costs, from its description.

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model  

## DAY 2: Data Pre-processing

Today we'll rewrite the products into a standard format.  
LLMs are great at this!


### 1 - Understand
- Activities:
    - Gather business reqauirements for the task
    - Identify performance criteria
        - Model and Business Centric Metrics
    - Understand the data: quantity, quality, format
    - Determine non-functionals
        - Cost constraints, scalability, latency
        - R&D(Research and Development) / build budget and implementation timeline

### 2 - Prepare
- Activities:
    - Research existing / non-LLM solutions
        - Potential baseline model
    - Compare relevant LLMs (week 4)
        - The basiscs, including context length, price and license
        - Benchmarks, Leaderboards and Arenas
        - Specialist scores for the task at hand
    - Curate data: clean, process and split (week 6) 

### 3 - Select
- Activities:
    - Choose LLM(s)
    - Experiment
    - Train and validate with curated data

### 4 - Customize
- Four techniques to optimize the performance of the model
    - Inference time:
        - Prompt: multishot prompting (week 2)
        - RAG: select relevant context and shovel into the prompt
        - Agents: decide llm to decide what to answer (tools - week 2)
    - Training Time:
        - Fine-tune: make data better to accomplish the task

### 5 - Productionize
- Activities:
    - Determine API between model and platform(s)
    - Identify model hosting and deployment architecture
    - Address scaling, monitoring, security, compliance, observability
    - Evals: measure the Business-Focused Metrics identified in step 1
    - Continuously retrain and measure performance
    

<table style="margin: 0; text-align: left;">
    <tr>
        <td>
            <h2 style="color:#181;">Business value of Data Pre-processing / Re-writing</h2>
            <span style="color:#181;">LLMs have made it simple to do something that was considered impossible only a few years ago: <i>turn unstructured data into structured data</i>.
            This approach can be applied to almost any business vertical, and it's similar to the advanced techniques
            we used on Week 5.</span>
        </td>
    </tr>
</table>

## This whole code can be skipe because it requires a lot of resources and at least $35s
- Ed will upload the structured data

In [ ]:
# !pip show litellm
# litellm-1.79.3

In [ ]:
from litellm import completion
from dotenv import load_dotenv
import json
from pricer.batch import Batch
from pricer.items import Item

load_dotenv(override=True)

# The next cell is where you choose Dataset

Use `LITE_MODE = True` for the free, fast version with training data size of 20,000

USe `LITE_MODE =  False` for the powerful, full version with training data size of 800,000

## For this lab

You can skip altogether and load the dataset from HuggingFace: $0

You can run pre-processing for the lite dataset: under $1

You can run pre-processing for the full dataset: $30

In [ ]:
LITE_MODE = True

In [ ]:
username = "ed-donner"
# to use dataset I "curated"
# username = "iara-santos"
dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"

train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f"Loaded {len(items):,} items")
print(items[0])

In [ ]:
items[2].id

In [ ]:
# Give every item an id
for index, item in enumerate(items):
    item.id = index

In [ ]:
SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

In [ ]:
print(items[0].full)

In [ ]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="groq/openai/gpt-oss-20b", reasoning_effort="low")

print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")


In [ ]:
# using a free llm ollama is less precise but it works
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="ollama/llama3.2", api_base="http://localhost:11434")
print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")


In [ ]:
# using Groq
MODEL = "openai/gpt-oss-20b"

In [ ]:
# json lines: each object one json line
def make_jsonl(item):
    body = {"model": MODEL, "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": item.full}], "reasoning_effort": "low"}
    line = {"custom_id": str(item.id), "method": "POST", "url": "/v1/chat/completions", "body": body}
    return json.dumps(line)

In [ ]:
items[0]

In [ ]:
make_jsonl(items[0])

In [ ]:
# create a file with the whole data we need to call groq to structure all the data we need
def make_file(start, end, filename):
    batch_file = filename
    with open(batch_file, "w") as f:
        for i in range(start, end):
            f.write(make_jsonl(items[i]))
            f.write("\n")

In [ ]:
make_file(0, 1000, "jsonl/0_1000.jsonl")

In [ ]:
import os
from groq import Groq

groq = Groq(api_key=os.environ.get("GROQ_API_KEY"))

In [ ]:
# create a file in groq for batch purposes
with open("jsonl/0_1000.jsonl", "rb") as f:
    response = groq.files.create(file=f, purpose="batch")
response

In [ ]:
# the file is in groq with the below id
file_id = response.id
file_id

In [ ]:
# create batches from the file id we got above
response = groq.batches.create(completion_window="24h", endpoint="/v1/chat/completions", input_file_id=file_id)
response

In [ ]:
# retrieve batches from the batches id we got above
result = groq.batches.retrieve(response.id)
result

In [ ]:
# collect the results
response = groq.files.content(result.output_file_id)
#  the structured data will be in response.content
response.write_to_file("jsonl/batch_results.jsonl")

In [ ]:
# iterate over items and get the ids to get the summary
with open("jsonl/batch_results.jsonl", "r") as f:
    for line in f:
        json_line = json.loads(line)
        id = int(json_line["custom_id"])
        summary = json_line["response"]["body"]["choices"][0]["message"]["content"]
        items[id].summary = summary


In [ ]:
print(items[0].full)

In [ ]:
print(items[1000].summary)

## I've put exactly this logic into a Batch class

- Divides items into groups of 1,000
- Kicks off batches for each
- Allows us to monitor and collect the results when complete

## COSTS

Using Groq, for me - this cost under $1 for the Lite dataset and under $30 for the big dataset

But you don't need to pay anything! In the next lab, you can load my pre-processed results

In [ ]:
Batch.create(items, LITE_MODE)

In [ ]:
Batch.run()

In [ ]:
Batch.fetch()

In [ ]:
for index, item in enumerate(items):
    if not item.summary:
        print(index)

In [ ]:
print(items[10234].summary)

In [ ]:
# Remove the fields that we don't need in the hub

for item in items:
    item.full = None
    item.id = None

## Push the final dataset to the hub

If lite mode, we'll only push the lite dataset

If full mode, we'll push both datasets (in case you decide to use lite later)

In [ ]:
username = "ed-donner"
full = f"{username}/items_full"
lite = f"{username}/items_lite"

if LITE_MODE:
    train = items[:20_000]
    val = items[20_000:21_000]
    test = items[21_000:]
    Item.push_to_hub(lite, train, val, test)
else:
    train = items[:800_000]
    val = items[800_000:810_000]
    test = items[810_000:]
    Item.push_to_hub(full, train, val, test)

    train_lite = train[:20_000]
    val_lite = val[:1_000]
    test_lite = test[:1_000]
    Item.push_to_hub(lite, train_lite, val_lite, test_lite)

## And here they are!

https://huggingface.co/datasets/ed-donner/items_lite

https://huggingface.co/datasets/ed-donner/items_full
